In [5]:
# ── Dependency Guard ───────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.3.3


In [6]:
# ── Reproducibility Header ────────────────────────────────────────────

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [7]:
# ── Rule-Based Prediction: Grid Position → DNF Prediction ────────────────

# Get race data from 2023 season (validation set)
# Using races from mid-season onwards
season = 2023
races_to_fetch = [10, 11, 12, 13, 14, 15]  # Races 10-22

predictions = []
actual_results = []
race_info = []

print(f"Fetching {len(races_to_fetch)} races from {season} season...")
print("=" * 70)

for race_round in races_to_fetch:
    try:
        session = fastf1.get_session(season, race_round, 'R')  # 'R' for race
        session.load()
        
        race_name = session.event['EventName']
        
        # Get results dataframe with grid and DNF status
        results = session.results
        
        # Identify DNF drivers (Status is not 'Finished')
        dnf_drivers = set(results[results['Status'] != 'Finished'].index)
        finished_drivers = set(results[results['Status'] == 'Finished'].index)
        
        # Apply rule: grid > 14 → predict DNF, grid <= 14 → predict finish
        for driver_idx in results.index:
            grid_pos = results.loc[driver_idx, 'GridPosition']
            
            # Skip if no valid grid position (sometimes NaN for reserve/safety car)
            if pd.isna(grid_pos):
                continue
            
            grid_pos = int(grid_pos)
            
            # Apply rule
            predicted_dnf = grid_pos > 14
            actual_dnf = driver_idx in dnf_drivers
            
            is_correct = predicted_dnf == actual_dnf
            
            predictions.append({
                'race': race_name,
                'round': race_round,
                'driver': driver_idx,
                'grid_position': grid_pos,
                'predicted_dnf': predicted_dnf,
                'actual_dnf': actual_dnf,
                'correct': is_correct
            })
        
        print(f"✓ Round {race_round}: {race_name} - {len(results)} drivers")
        
    except Exception as e:
        print(f"✗ Round {race_round}: Error - {str(e)}")

# Convert to DataFrame for analysis
predictions_df = pd.DataFrame(predictions)

# Calculate metrics
total_predictions = len(predictions_df)
correct_predictions = predictions_df['correct'].sum()
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

print("=" * 70)
print(f"\n📊 RULE-BASED PREDICTION RESULTS")
print(f"{'─' * 70}")
print(f"Rule: If GridPosition > 14 → Predict DNF | Otherwise → Predict Finish")
print(f"{'─' * 70}")
print(f"Total Predictions:  {total_predictions}")
print(f"Correct:            {correct_predictions}")
print(f"Incorrect:          {total_predictions - correct_predictions}")
print(f"\n🎯 ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'─' * 70}")

# Breakdown by outcome
print(f"\nDetailed Breakdown:")
print(f"  - Correctly predicted DNF:    {predictions_df[(predictions_df['predicted_dnf'] == True) & (predictions_df['correct'] == True)].shape[0]}")
print(f"  - Correctly predicted Finish: {predictions_df[(predictions_df['predicted_dnf'] == False) & (predictions_df['correct'] == True)].shape[0]}")
print(f"  - False positives (DNF):      {predictions_df[(predictions_df['predicted_dnf'] == True) & (predictions_df['correct'] == False)].shape[0]}")
print(f"  - False negatives (Finish):   {predictions_df[(predictions_df['predicted_dnf'] == False) & (predictions_df['correct'] == False)].shape[0]}")


Fetching 6 races from 2023 season...


core           INFO 	Loading data for British Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '44', '81', '63', '11', '14', '23', '16', '55', '2', '77', '27', '18', '24', '22', '21', '10', '20', '31']
core           INFO 	Loading data for Hungarian Grand Prix - 

✓ Round 10: British Grand Prix - 20 drivers


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '11', '44', '81', '63', '16', '55', '14', '18', '23', '77', '3', '27', '22', '24', '20', '2', '31', '10']
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✓ Round 11: Hungarian Grand Prix - 20 drivers


...

✓ Round 13: Dutch Grand Prix - 20 drivers


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 06:25.888000 before the recorded end of the session.
core        WARNING 	Driver 11 completed the race distance 06:19.824000 before the recorded end of the session.
core        WARNING 	Driver 55 completed the race distance 06:14.695000 before the recorded end of the session.
core        WARNING 	Driver 16 completed the race distance 06:14.511000 before the recorded end of the session.
core        WARNING 	Driver 63 completed the race distance 06:07.860000 before the recorded end of the session.
core        WARNING 	Driver 44 completed the race distance 05:48.209000 before the recorded end of the session.
core        WARNING 	Driver 23 completed the race distance 05:40.782000 before the recorded end of 

✓ Round 14: Italian Grand Prix - 20 drivers


core        WARNING 	No lap data for driver 18
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['55', '4', '44', '16', '1', '10', '81', '11', '40', '20', '23', '24', '27', '2', '14', '63', '77', '31', '22', '18']


✓ Round 15: Singapore Grand Prix - 20 drivers

📊 RULE-BASED PREDICTION RESULTS
──────────────────────────────────────────────────────────────────────
Rule: If GridPosition > 14 → Predict DNF | Otherwise → Predict Finish
──────────────────────────────────────────────────────────────────────
Total Predictions:  119
Correct:            80
Incorrect:          39

🎯 ACCURACY: 0.6723 (67.23%)
──────────────────────────────────────────────────────────────────────

Detailed Breakdown:
  - Correctly predicted DNF:    11
  - Correctly predicted Finish: 69
  - False positives (DNF):      23
  - False negatives (Finish):   16


The acuarcy of the method was 67.23% for the lab 2 we need to at least get that accurate.